# CD CosMx Resection: LR Module Scoring and LOWESS Scatter (Ext Fig 10)

Downstream analysis of LIANA results across all 12 resection samples.
1. Aggregate pre-computed LIANA results from parquet files
2. Categorize LR pairs into 4 functional modules:
   - Ag-Pres/Treg Destab, Pro-Inflammatory, Immunomodulatory, Pro-Fibrotic/ECM
3. FOV-level module z-score vs Late−Early Treg / OGN+Fib / CD9+ Mac proportions
4. LOWESS scatter plots

In [ ]:
# ── Paths ──
DATA_DIR   = "/path/to/cosmx_data/cd_resection"
OUTPUT_DIR = DATA_DIR + "/combined/liana_modules"

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns
import os
import scanpy.external as sce
import liana as li
from scipy.spatial import distance_matrix

In [ ]:
lrdata_CD_B4=sc.read_h5ad('/path/to/cosmx_data/cd_6k_wtx/whole_trans/Processed_merged/h5ad_obj/lr_liana_CD_B4_1_nn_db.h5ad')

In [ ]:
lrdata_CD_B5=sc.read_h5ad('/path/to/cosmx_data/cd_6k_wtx/whole_trans/Processed_merged/h5ad_obj/lr_liana_CD_B5_1_nn_db.h5ad')

In [ ]:
lrdata_CD_B5_3=sc.read_h5ad('/path/to/cosmx_data/cd_6k_wtx/whole_trans/Processed_merged/h5ad_obj/lr_liana_CD_B5_3_nn_db.h5ad')

In [ ]:
lrdata_CD_B4_3=sc.read_h5ad('/path/to/cosmx_data/cd_6k_wtx/whole_trans/Processed_merged/h5ad_obj/lr_liana_CD_B4_3_nn_db.h5ad')

In [ ]:
morans_sorted_CD_B4 = lrdata_CD_B4.var.sort_values("morans", ascending=False)

In [ ]:
# Pro-Fibrotic/ECM Remodeling
profibrotic_ecm = [
        'TIMP2^MMP2',
        'COL3A1^DDR2',
        'COL17A1^ITGB1',
        'COL9A1^ITGA2',
        'COL27A1^ITGB1',
        'TGM2^SDC4'
]

# Pro-Inflammatory/Immune Activation
proinflammatory = [
        'TNFSF13^TNFRSF1A',
        'TNFSF13^TNFRSF14',
        'IL6^IL6ST',
        'IL1A^IL1RAP',
        'CCL2^CCR3',
        'CCL19^CCR7',
        'CCL25^CCR9',
        'CCL25^ACKR4',
        'CCL11^ACKR4'
]

# Antigen Presentation
ag_presentation_treg = [
'HLA-DRA^CD63',
 'HLA-DRB1^CD81',
 'HLA-DRA^CD81',
 'ICAM1^MSN',
 'ITGB2^CD226',
 'ICAM1^SPN',
 'PLA2G2A^ITGB1',
 'CD99^CD81',
 'CXCL14^CXCR4',
 'HLA-DRA^LAG3',
 'SIRPA^CD69',
 'CCL4^CCR5',
]

# Immunomodulatory (Treg-related)
immunomodulatory = [
        'LGALS1^PTPRC',
        'IL12A^CD28',
        'CD58^CD2',
        'PTPRC^CD22',
        'CD22^PTPRC'
]


# All notable interactions combined
all_notable = (profibrotic_ecm  + proinflammatory + 
               ag_presentation_treg + immunomodulatory )
print(f"\nTotal: {len(all_notable)}")  

In [ ]:
# Add a column that annotates each LR interaction with the function group defined above (cell 40)
def lr_function(lr):
    if lr in profibrotic_ecm:
        return 'Pro-Fibrotic/ECM'
    if lr in proinflammatory:
        return 'Pro-Inflammatory'
    if lr in ag_presentation_treg:
        return 'Ag Presentation/Treg'
    if lr in immunomodulatory:
        return 'Immunomodulatory'
    return 'other'


In [ ]:
# apply to morans_sorted index (assumes lr strings like 'GENE^GENE')
morans_sorted_CD_B4['function'] = morans_sorted_CD_B4.index.map(lr_function)
morans_sorted_CD_B4[morans_sorted_CD_B4['function']!='other']
morans_sorted_CD_B4['sample'] = morans_sorted_CD_B4.index.map(lr_function)

In [ ]:
import os
import pandas as pd
import anndata as ad
import gc

def concatenate_fov_var_per_sample(base_dir, sample_folders, output_suffix='_liana_var_combined.parquet'):
    """
    For each sample, concatenate .var from all FOV h5ad files in the liana folder and save.
    """
    
    for sample in sample_folders:
        print(f"\nProcessing sample: {sample}")
        
        liana_path = os.path.join(base_dir, sample, "Processed", "liana")
        output_path = os.path.join(base_dir, sample, "Processed", f"{sample}{output_suffix}")
        
        # Get all h5ad files in liana directory
        files = [f for f in os.listdir(liana_path) if f.endswith('.h5ad')]
        
        print(f"  Found {len(files)} FOV h5ad files")
        
        try:
            var_list = []
            for i, file in enumerate(files, 1):
                file_path = os.path.join(liana_path, file)
                
                # Read h5ad in backed mode
                adata = ad.read_h5ad(file_path, backed='r')
                
                # Extract var dataframe
                var_df = adata.var.copy()
                
                # Add FOV identifier
                fov_id = os.path.splitext(file)[0]
                var_df['fov'] = fov_id
                
                var_list.append(var_df)
                
                # Close file handle
                adata.file.close()
                del adata, var_df
                
                if i % 10 == 0:
                    print(f"    Loaded {i}/{len(files)} files...")
                    gc.collect()
            
            # Concatenate all FOV var dataframes for this sample
            print(f"  Concatenating {len(var_list)} dataframes...")
            sample_combined = pd.concat(var_list, ignore_index=False)
            
            # Add sample identifier
            sample_combined['sample'] = sample
            
            # Save the combined sample data
            sample_combined.to_parquet(output_path)
            print(f"  ✓ Saved to {output_path}")
            print(f"    Shape: {sample_combined.shape}")
            
            # Clean up
            del var_list, sample_combined
            gc.collect()
            
        except Exception as e:
            print(f"  ✗ Error processing {sample}: {e}")
            gc.collect()



In [ ]:
def concatenate_all_samples(base_dir, sample_folders, output_suffix='_liana_var_combined.parquet', 
                           final_output='all_samples_liana_var_combined.parquet'):
    """
    Concatenate all sample-level combined var files into one final file.
    """
    print(f"\nConcatenating all {len(sample_folders)} samples...")
    
    all_samples_list = []
    
    for i, sample in enumerate(sample_folders, 1):
        sample_file = os.path.join(base_dir, sample, "Processed", f"{sample}{output_suffix}")
        
        df = pd.read_parquet(sample_file)
        all_samples_list.append(df)
        
        print(f"  Loaded {i}/{len(sample_folders)}: {sample} (shape: {df.shape})")
        
        if i % 10 == 0:
            gc.collect()
    
    print(f"\nConcatenating {len(all_samples_list)} samples...")
    final_combined = pd.concat(all_samples_list, ignore_index=False)
    
    # Save final combined file
    final_output_path = os.path.join(base_dir, final_output)
    final_combined.to_parquet(final_output_path)
    
    print(f"✓ Final combined file saved to {final_output_path}")
    print(f"  Final shape: {final_combined.shape}")
    
    del all_samples_list
    gc.collect()
    
    return final_combined


In [ ]:
sample_list =[
#'Cos1_6KDC_strict18888_A5_0715_11_08_2025_14_21_12_760',                
'Cos1_6KDC_strict43563_SB_A1_091725BP6_09_10_2025_13_54_55_807',          
'Cos1_6KDC_strict43563_SB_A8_091725BP5_09_10_2025_13_41_29_437',       
'Cos1_6KDC_strictCD_R1_B4_0715_11_08_2025_14_20_23_133',     
'Cos2_6KDC_strict18888_A10_0715_Alan_data_test_11_08_2025_14_25_11_455',  
'Cos2_6KDC_strict18888_A6_0715_Alan_Test_11_08_2025_14_23_09_489',  
'Cos3_6KDC_strict11808_A4_0716_11_08_2025_14_19_48_615',                  
'Cos3_6KDC_strictCD_R1_B5_0716_11_08_2025_14_17_56_277',                
'Cos4_6KDC_strict11808_A4_0716_14_10_2025_12_34_38_727',
'Cos4_6KDC_strictCD_R1_B7_0716_11_08_2025_14_18_41_738',
'Cos5_6KDC_strict43563_SB_A11_091725BP7_09_10_2025_13_42_42_775',         
'Cos5_6KDC_strict43563_SB_A4_091725BP8_09_10_2025_13_43_50_952'  
]

In [ ]:

# Usage
base_dir = '/path/to/cosmx_data/cd_resection'
sample_folders = sample_list

# Step 1: Concatenate FOV .var from h5ad files within each sample
concatenate_fov_var_per_sample(base_dir, sample_folders)


In [ ]:

# Step 2: Concatenate all sample files into one final file
final_df = concatenate_all_samples(base_dir, sample_folders)

In [ ]:
final_df=pd.read_parquet('/path/to/cosmx_data/cd_resection/all_samples_liana_var_combined.parquet')

In [ ]:
comb_lst = {}
for i in sample_list: 
    if 'SB' in i:
        patient = i.split('strict')[1].split('_')[0]
        section = i.split('strict')[1].split('_')[2]
        comb = patient+'_'+section
    elif '11808' in i:
        patient = i.split('strict')[1].split('_')[0]
        section = i.split('strict')[1].split('_')[1]
        machine = i.split('_')[0]
        comb = patient+'_'+section+'_'+machine
    else:
        patient = i.split('strict')[1].split('_')[0]
        section = i.split('strict')[1].split('_')[1]
        comb = patient+'_'+section
    comb_lst[i]=comb

In [ ]:
final_df['patient_sec'] = final_df['sample'].map(comb_lst)

In [ ]:
final_df_notable=final_df[final_df.index.isin(all_notable)]

In [ ]:
final_df_notable['function'] = final_df_notable.index.map(lr_function)

In [ ]:
final_df_fib=final_df_notable[final_df_notable['function']=='Pro-Fibrotic/ECM']
final_df_ag=final_df_notable[final_df_notable['function']=='Ag Presentation/Treg']
final_df_mod=final_df_notable[final_df_notable['function']=='Immunomodulatory']
final_df_inf=final_df_notable[final_df_notable['function']=='Pro-Inflammatory']

In [ ]:
final_df_fib=final_df_fib.sort_values('mean',ascending=False)
final_df_ag=final_df_ag.sort_values('mean',ascending=False)
final_df_mod=final_df_mod.sort_values('mean',ascending=False)
final_df_inf=final_df_inf.sort_values('mean',ascending=False)

In [ ]:

# Convert fields
final_df_notable['fov'] = final_df_notable['fov'].astype(str)
final_df_notable['patient_sec'] = final_df_notable['patient_sec'].astype(str)
final_df_notable['function'] = final_df_notable['function'].astype(str)

# Compute mean LR score ("mean" column) per (patient, fov, function)
module_scores = (
    final_df_notable
    .groupby(['patient_sec', 'fov', 'function'])['mean']
    .mean()
    .reset_index()
)

# Pivot so each module becomes its own column
module_scores = module_scores.pivot(
    index=['patient_sec', 'fov'],
    columns='function',
    values='mean'
).reset_index()


In [ ]:
modules = [
    "Ag Presentation/Treg",
    "Immunomodulatory",
    "Pro-Fibrotic/ECM",
    "Pro-Inflammatory"
]


In [ ]:
module_scores[modules] = module_scores[modules].apply(pd.to_numeric, errors='coerce')

In [ ]:
module_scores[modules].describe()
module_scores[modules].std()

In [ ]:
module_scores.to_csv('/path/to/cosmx_data/cd_resection/combined/liana_fov_sample_module_scores_new_treg.csv',index=False)

In [ ]:
import numpy as np

df_z = module_scores.copy()

# make sure module columns are numeric
df_z[modules] = df_z[modules].apply(pd.to_numeric, errors="coerce")

# compute mean/std *ignoring NaNs*
means = df_z[modules].mean(skipna=True)
stds  = df_z[modules].std(skipna=True, ddof=0)

# z-score each column; NaNs stay NaN
for col in modules:
    if stds[col] == 0 or np.isnan(stds[col]):
        # if no variation or all NaN → just set z = 0 or leave as is
        df_z[col] = 0.0
    else:
        df_z[col] = (df_z[col] - means[col]) / stds[col]


In [ ]:
df_z_orig=df_z.copy()

In [ ]:
df_temp = spot_anno.copy()

# Filter to only rows where early_late is Early or Late
df_temp = df_temp[df_temp['early_late'].isin(['Early Treg', 'Late Treg'])]
#df_temp = df_temp[df_temp['cell_category']=='T']

# Compute counts per FOV × early_late
counts = (
    df_temp
    .groupby(['patient_sec','fov', 'early_late'])
    .size()
    .unstack(fill_value=0)
)

# Compute total per FOV
counts['total'] = counts.sum(axis=1)

# Compute proportions
counts['prop_early'] = counts['Early Treg'] / counts['total']
counts['prop_late']  = counts['Late Treg']  / counts['total']

# Compute difference (Late - Early)
counts['late_minus_early'] = counts['prop_late'] - counts['prop_early']

counts =counts.reset_index()
counts

In [ ]:
df_z_plot = df_z.copy()
df_z_plot['fov']=[i.split('fov_')[1] for i in df_z_plot['fov'].tolist() ]
df_z_plot

In [ ]:
df_z_plot = df_z_plot.merge(counts, how = 'left', left_on = ['patient_sec','fov'], right_on = ['patient_sec','fov'])

In [ ]:
df_temp = spot_anno.copy()

# Filter to only rows where early_late is Early or Late
#df_temp = df_temp[df_temp['cell_category']=='Connective']
df_temp = df_temp[df_temp['cell_type_general'].isin(['OGN+RSPO3+ Fib', 'Myofibroblast'])]

# Compute counts per FOV × early_late
counts = (
    df_temp
    .groupby(['patient_sec','fov', 'cell_type_general'])
    .size()
    .unstack(fill_value=0)
)

# Compute total per FOV
counts['total'] = counts.sum(axis=1)


# Compute proportions
counts['prop_ogn'] = counts['OGN+RSPO3+ Fib'] / counts['total']
counts['prop_myo'] = counts['Myofibroblast'] / counts['total']

# Compute difference (Late - Early)
counts['ogn_minus_myofib'] = counts['prop_ogn'] - counts['prop_myo']

counts =counts.reset_index()
counts

In [ ]:
df_z_plot = df_z_plot.merge(counts[['patient_sec','fov','prop_ogn','prop_myo','ogn_minus_myofib']], how = 'left', left_on = ['patient_sec','fov'], right_on = ['patient_sec','fov'])

In [ ]:
df_temp = spot_anno.copy()

# Filter to only rows where early_late is Early or Late
#df_temp = df_temp[df_temp['cell_category']=='Connective']
df_temp = df_temp[df_temp['cell_type_short'].isin(['CD9+ Mac', 'Mo-Mac'])]

# Compute counts per FOV × early_late
counts = (
    df_temp
    .groupby(['patient_sec','fov', 'cell_type_short'])
    .size()
    .unstack(fill_value=0)
)

# Compute total per FOV
counts['total'] = counts.sum(axis=1)


# Compute proportions
counts['prop_cd9'] = counts['CD9+ Mac'] / counts['total']
counts['prop_mo'] = counts['Mo-Mac'] / counts['total']

# Compute difference (Late - Early)
counts['cd9_minus_momac'] = counts['prop_cd9'] - counts['prop_mo']

counts =counts.reset_index()
counts

In [ ]:
df_z_plot = df_z_plot.merge(counts[['patient_sec','fov','prop_cd9','prop_mo','cd9_minus_momac']], how = 'left', left_on = ['patient_sec','fov'], right_on = ['patient_sec','fov'])

In [ ]:

# Compute counts per FOV × early_late
counts_tot = pd.DataFrame(
    spot_anno
    .groupby(['patient_sec','fov'])
    .size()
)
counts_tot.columns =['total']
counts_tot

counts = pd.DataFrame(
    df_temp
    .groupby(['patient_sec','fov', 'cell_type_short'])
    .size()
)
counts = counts.droplevel('cell_type_short')
counts.columns = ['B']
counts = counts_tot.merge(
    counts,   # only keys
    how='right',
    left_index=True, right_index=True
)


# Compute proportions
counts['prop_b'] = counts['B'] / counts['total']

counts =counts.reset_index()
counts

In [ ]:
df_z_plot = df_z_plot.merge(counts[['patient_sec','fov','prop_b']], how = 'left', left_on = ['patient_sec','fov'], right_on = ['patient_sec','fov'])

In [ ]:
df_z_plot = df_z_plot[['patient_sec', 'fov', 'Ag Presentation/Treg', 'Immunomodulatory',
       'Pro-Fibrotic/ECM', 'Pro-Inflammatory', 'prop_early', 'prop_late', 'late_minus_early',
                       'prop_ogn', 'prop_myo', 'ogn_minus_myofib','prop_cd9','prop_mo','cd9_minus_momac','prop_b']]
df_z_plot

In [ ]:
from scipy.stats import spearmanr    
from statsmodels.nonparametric.smoothers_lowess import lowess
def plot_lowess_scatter(
    df,
    x_col,
    y_col,
    hue_col="patient_sec",
    figsize=(6, 3.8),
    x_label=None,
    y_label=None,
    title=None,
    point_size=20,
    alpha=0.6,
    lowess_frac=0.4,
    annotate=True,
    va='bottom',
    ha='right',
    va_pos=0.95,
    ha_pos=0.05,
):
    """
    Scatter plot with LOWESS (non-parametric) trend line.
    Designed to avoid imposing linear structure.
    """


    # Filter valid rows for LOWESS
    mask = df[x_col].notna() & df[y_col].notna()
    df_valid = df.loc[mask].copy()

    x = df_valid[x_col].values
    y = df_valid[y_col].values
    rho, pval = spearmanr(x, y)
    plt.figure(figsize=figsize)

    # Scatter plot
    sns.scatterplot(
        data=df,
        x=x_col,
        y=y_col,
        hue=hue_col,
        s=point_size,
        alpha=alpha,
        edgecolor=None
    )

    # LOWESS smoothing (only if enough points)
    '''
    if len(df_valid) >= 5 and np.std(x) > 0:
        lowess_fit = lowess(
            y,
            x,
            frac=lowess_frac,
            return_sorted=True
        )

        plt.plot(
            lowess_fit[:, 0],
            lowess_fit[:, 1],
            color="black",
            linewidth=2,
            alpha=0.85,
            label="LOWESS trend"
        )
    '''
    # Optional annotation (purely descriptive)
    if annotate:
        plt.text(
            va_pos,
            ha_pos,
            f"Spearman ρ = {rho:.2f}\nP = {pval:.2e}",
            #f"LOWESS frac = {lowess_frac}",
            transform=plt.gca().transAxes,
            va=va,
            ha=ha,
            fontsize=11,
            bbox=dict(facecolor='white', edgecolor='gray', alpha=0.6)
        )

    # Legend outside
    plt.legend(
        title="Resection",
        bbox_to_anchor=(1.05, 1),
        loc="upper left"
    )

    # Labels
    plt.xlabel(x_label if x_label else x_col)
    plt.ylabel(y_label if y_label else y_col)

    # Title
    if title is not None:
        plt.title(title)

    plt.tight_layout()
    plt.show()


In [ ]:
plot_lowess_scatter(
    df=df_z_plot,
    x_col="Ag Presentation/Treg",
    y_col="late_minus_early",
    x_label="Ag Pres/Treg Destab LR Score",
    y_label="Late − Early Treg",
    title="",
    lowess_frac=0.35,
    va_pos=0.51,
    ha_pos=0.87,
)


In [ ]:


plot_lowess_scatter(
    df=df_z_plot,
    x_col="Pro-Inflammatory",
    y_col="late_minus_early",
    x_label="Pro-Inflammatory LR Score",
    y_label="Late − Early Treg",
    title="",
    lowess_frac=0.3,

)


In [ ]:


plot_lowess_scatter(
    df=df_z_plot,
    x_col="Immunomodulatory",
    y_col="late_minus_early",
    x_label="Immunomodulatory LR Score",
    y_label="Late − Early Treg",
    title="",
    lowess_frac=0.3,
            va_pos=0.97,
    ha_pos=0.87,
)


In [ ]:
plot_lowess_scatter(
    df=df_z_plot,
    x_col="Ag Presentation/Treg",
    y_col="ogn_minus_myofib",
    x_label="Ag Pres/Treg Destab LR Score",
    y_label="OGN+Fib - Myofib",
    title="",
    lowess_frac=0.3,
        va_pos=0.51,
    ha_pos=0.87,
)


In [ ]:


plot_lowess_scatter(
    df=df_z_plot,
    x_col="Pro-Fibrotic/ECM",
    y_col="ogn_minus_myofib",
    x_label="Pro-Fibrotic/ECM LR Score",
    y_label="OGN+Fib - Myofib",
    title="",
    lowess_frac=0.3,
        va_pos=0.97,
    ha_pos=0.87,
)


In [ ]:

plot_lowess_scatter(
    df=df_z_plot,
    x_col="Pro-Inflammatory",
    y_col="ogn_minus_myofib",
    x_label="Pro-Inflammatory LR Score",
    y_label="OGN+Fib - Myofib",
    title="",
    lowess_frac=0.3,
        va_pos=0.97,
    ha_pos=0.87,
)


In [ ]:
final_df_inf2 = final_df_inf.copy()
df_z2 = df_z_plot.copy()

# Ensure df_z fov is string and add prefix
df_z2['fov'] = df_z2['fov'].astype(str)
df_z2['fov'] = 'fov_' + df_z2['fov']

In [ ]:
final_df_inf_ct_prop = final_df_inf.merge(df_z2[['patient_sec','fov','ogn_minus_myofib','late_minus_early']], how = 'right', on = ['patient_sec','fov'])
final_df_inf_ct_prop['LR'] = [str(i)+'^'+str(j) for i,j in zip (final_df_inf_ct_prop['ligand'].tolist(),final_df_inf_ct_prop['receptor'].tolist())]
final_df_inf_ct_prop

In [ ]:
final_df_fib_ct_prop = final_df_fib.copy()
final_df_fib_ct_prop = final_df_fib_ct_prop.merge(df_z2[['patient_sec','fov','ogn_minus_myofib','late_minus_early']], how = 'right', on = ['patient_sec','fov'])
final_df_fib_ct_prop['LR'] = [str(i)+'^'+str(j) for i,j in zip (final_df_fib_ct_prop['ligand'].tolist(),final_df_fib_ct_prop['receptor'].tolist())]
final_df_fib_ct_prop

In [ ]:
final_df_ag_ct_prop = final_df_ag.copy()
final_df_ag_ct_prop = final_df_ag_ct_prop.merge(df_z2[['patient_sec','fov','ogn_minus_myofib','late_minus_early']], how = 'right', on = ['patient_sec','fov'])
final_df_ag_ct_prop['LR'] = [str(i)+'^'+str(j) for i,j in zip (final_df_ag_ct_prop['ligand'].tolist(),final_df_ag_ct_prop['receptor'].tolist())]
final_df_ag_ct_prop

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

def lr_pearson_all_in_one(
    df,
    lr_col="LR",
    score_col="mean",
    outcome_col="late_minus_early",
    patient_col="patient_sec",
    min_n_fov=3,
    min_n_patient=2
):
    """
    Returns ONE dataframe containing both:
      - overall Pearson per LR pair
      - per-patient Pearson per LR pair
      - mean Pearson across patients (repeated on each row of that LR)

    Output columns:
      lr_pair
      level               ('overall' or 'patient')
      patient             (NaN for overall)
      pearson_r
      p_value
      n_fovs
      mean_r_across_patients  (same repeated for all rows of that LR)
      n_patients_used
    """

    df = df.copy()

    # Clean LR names
    df = df[df[lr_col].notna()]
    df[lr_col] = df[lr_col].astype(str)
    df = df[(df[lr_col] != "nan") & (df[lr_col] != "nan^nan")]

    # Ensure numeric
    df[score_col] = pd.to_numeric(df[score_col], errors="coerce")
    df[outcome_col] = pd.to_numeric(df[outcome_col], errors="coerce")

    # Z-score LR score within LR pair
    def zscore_group(subdf):
        vals = subdf[score_col]
        m = vals.mean(skipna=True)
        s = vals.std(skipna=True, ddof=0)
        subdf[score_col + "_z"] = 0.0 if s == 0 or np.isnan(s) else (vals - m) / s
        return subdf

    df = df.groupby(lr_col, group_keys=False).apply(zscore_group)
    z_col = score_col + "_z"

    rows = []

    # Loop LR pairs
    for lr_name, subdf in df.groupby(lr_col):

        sub_valid = subdf[[z_col, outcome_col, patient_col]].dropna()
        n_all = len(sub_valid)

        # -------- Overall Pearson --------
        if (
            n_all >= min_n_fov
            and sub_valid[z_col].nunique() > 1
            and sub_valid[outcome_col].nunique() > 1
        ):
            r_all, p_all = pearsonr(sub_valid[z_col], sub_valid[outcome_col])
        else:
            r_all, p_all = np.nan, np.nan

        # Compute per-patient Pearson
        patient_rs = []

        for patient_id, sub_pat in sub_valid.groupby(patient_col):
            n_pat = len(sub_pat)
            if (
                n_pat >= min_n_fov
                and sub_pat[z_col].nunique() > 1
                and sub_pat[outcome_col].nunique() > 1
            ):
                r_pat, p_pat = pearsonr(sub_pat[z_col], sub_pat[outcome_col])
            else:
                r_pat, p_pat = np.nan, np.nan

            patient_rs.append(r_pat)

            rows.append({
                "lr_pair": lr_name,
                "level": "patient",
                "patient": patient_id,
                "pearson_r": r_pat,
                "p_value": p_pat,
                "n_fovs": n_pat,
                # filled later
                "mean_r_across_patients": np.nan,
                "n_patients_used": np.nan
            })

        # Compute mean across patients (ignoring NaN)
        valid_patient_rs = [v for v in patient_rs if np.isfinite(v)]
        if len(valid_patient_rs) >= min_n_patient:
            mean_r = float(np.mean(valid_patient_rs))
            n_patients_used = len(valid_patient_rs)
        else:
            mean_r = np.nan
            n_patients_used = len(valid_patient_rs)

        # Add overall row
        rows.append({
            "lr_pair": lr_name,
            "level": "overall",
            "patient": np.nan,
            "pearson_r": r_all,
            "p_value": p_all,
            "n_fovs": n_all,
            "mean_r_across_patients": mean_r,
            "n_patients_used": n_patients_used
        })

    # Convert to dataframe
    out = pd.DataFrame(rows)

    # Fill mean_r_across_patients & n_patients_used for ALL rows of each LR
    out["mean_r_across_patients"] = (
        out.groupby("lr_pair")["mean_r_across_patients"].transform(
            lambda x: x.ffill().bfill()
        )
    )
    out["n_patients_used"] = (
        out.groupby("lr_pair")["n_patients_used"].transform(
            lambda x: x.ffill().bfill()
        )
    )

    return out


In [ ]:
treg_inf = lr_pearson_all_in_one(
    df=final_df_inf_ct_prop,
    lr_col="LR",
    score_col="mean",
    outcome_col="late_minus_early",
    patient_col="patient_sec"
)

treg_inf_sum = treg_inf[treg_inf["level"] == "overall"].rename(
    columns={"pearson_r": "pearson_r_overall",
             "p_value": "p_value_overall"}
)

treg_inf_pat = treg_inf[treg_inf["level"] == "patient"].rename(
    columns={"pearson_r": "pearson_r_patient"}
)


In [ ]:

ogn_ag = lr_pearson_all_in_one(
    df=final_df_ag_ct_prop,
    lr_col="LR",
    score_col="mean",
    outcome_col="ogn_minus_myofib",
    patient_col="patient_sec"
)

ogn_ag_sum = ogn_ag[ogn_ag["level"] == "overall"].rename(
    columns={"pearson_r": "pearson_r_overall",
             "p_value": "p_value_overall"}
)

ogn_ag_pat = ogn_ag[ogn_ag["level"] == "patient"].rename(
    columns={"pearson_r": "pearson_r_patient"}
)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def plot_lr_with_patient_jitter(
    summary_df,
    per_patient_df,
    xlabel,
    ylabel,
    x_col="lr_pair",
    y_col="pearson_r_overall",      # column in summary_df
    patient_y_col="pearson_r_patient",  # column in per_patient_df
    p_col="p_value_overall",
    top_n=20,
    bar_color="#5C7AFF",
    dot_color="#333333",
    figsize=(3.8, 4),
    bar_width=0.6
):
    """
    Plot LR associations with:
      - one bar (or point) per LR for overall correlation
      - jittered dots for per-patient correlations
      - p-value stars above each LR

    summary_df: per-LR summary (overall correlations)
    per_patient_df: per-LR-per-patient correlations
    """

    # 1. Prep summary data: drop NaNs, sort, take top_n
    d_sum = summary_df.dropna(subset=[y_col]).copy()
    d_sum = d_sum.sort_values(y_col, ascending=False).head(top_n)

    # x order (strongest on the left)
    order = d_sum[x_col].astype(str).values

    # 2. Filter per-patient DF to these LR pairs
    d_pat = per_patient_df.copy()
    d_pat[x_col] = d_pat[x_col].astype(str)
    d_pat = d_pat[d_pat[x_col].isin(order)]

    # Align categorical ordering
    d_sum[x_col] = pd.Categorical(d_sum[x_col], categories=order, ordered=True)
    d_pat[x_col] = pd.Categorical(d_pat[x_col], categories=order, ordered=True)

    # 3. Set up figure
    plt.figure(figsize=figsize)
    ax = plt.gca()

    # 4. Jittered dots for per-patient correlations
    sns.stripplot(
        data=d_pat,
        x=x_col,
        y=patient_y_col,
        order=order,
        color=dot_color,
        alpha=0.7,
        size=3.5,
        jitter=True,
        ax=ax
    )

    # 5. Overlay bar (or big point) for overall correlation
    # Here we do bars; you could replace with points if you prefer
    sns.barplot(
        data=d_sum,
        x=x_col,
        y=y_col,
        order=order,
        color=bar_color,
        width=bar_width,
        alpha=0.5,    # slightly transparent so dots are visible
        ax=ax
    )

    # Axes labels & title
    plt.xticks(rotation=90, fontsize=9)
    plt.yticks(fontsize=9)
    plt.xlabel(xlabel, fontsize=10)
    plt.ylabel(ylabel, fontsize=10)

    # Horizontal zero line
    ax.axhline(0, color="gray", linestyle="--", linewidth=1)

    # 6. Compute y-limits based on both overall + patient correlations
    all_vals = []
    if len(d_sum) > 0:
        all_vals.append(d_sum[y_col].values)
    if len(d_pat) > 0:
        all_vals.append(d_pat[patient_y_col].values)

    all_vals = np.concatenate([v for v in all_vals if len(v) > 0]) if all_vals else np.array([0.0])
    max_y = np.nanmax(all_vals)
    min_y = np.nanmin(all_vals)

    y_range = max_y - min_y if np.isfinite(max_y - min_y) and (max_y - min_y) > 0 else 0.1
    y_buffer = 0.15 * y_range  # 15% buffer

    ax.set_ylim(min_y - y_buffer, max_y + y_buffer)

    # 7. p-value star helper
    def p_to_stars(p):
        if not np.isfinite(p):
            return ""
        if p <= 0.001:
            return "***"
        elif p <= 0.01:
            return "**"
        elif p <= 0.05:
            return "*"
        else:
            return ""

    # 8. Add stars above bars (based on summary_df p-values)
    max_abs = np.nanmax(np.abs(all_vals)) if len(all_vals) else 0.1
    star_offset = 0.03 * y_range

    for patch, (_, row) in zip(ax.patches[-len(d_sum):], d_sum.iterrows()):
        height = patch.get_height()
        if not np.isfinite(height):
            continue
        x = patch.get_x() + patch.get_width() / 2
        y = height + star_offset
        stars = p_to_stars(row[p_col])
        if stars:
            ax.text(
                x, y,
                stars,
                ha="center",
                va="bottom",
                fontsize=10,
                fontweight="bold"
            )

    plt.tight_layout()
    plt.show()


In [ ]:

plot_lr_with_patient_jitter(
    summary_df=ogn_ag_sum,
    per_patient_df=ogn_ag_pat,
    xlabel="Ag/Treg Destab LRs",
    ylabel="Spearman w/ OGN+Fib–Myofib Δ",
    x_col="lr_pair",
    y_col="pearson_r_overall",
    patient_y_col="pearson_r_patient",
    p_col="p_value_overall",
    top_n=15,
    bar_color="#c2c2c2",
    dot_color="#91a3ce",
   figsize=(3.5, 4)
)


In [ ]:
combined_obs=pd.read_parquet('/path/to/cosmx_data/cd_resection/combined/all_samples.parquet')
combined_obs

In [ ]:

global_categories = ['B', 'Connective', 'Endothelial', 'Epithelial', 'Granulocyte',
                     'Myeloid', 'NK', 'Neuron', 'Plasma', 'T','Cycling']
global_palette = [
    (0.12156862745098039, 0.4666666666666667, 0.7058823529411765),  # B
    (0.6823529411764706, 0.7803921568627451, 0.9098039215686274),  # Connective
    (1.0, 0.4980392156862745, 0.054901960784313725),              # Endothelial
    (1.0, 0.7333333333333333, 0.47058823529411764),               # Epithelial
    (0.17254901960784313, 0.6274509803921569, 0.17254901960784313),# Granulocyte
    (0.596078431372549, 0.8745098039215686, 0.5411764705882353),  # Myeloid
    (0.8392156862745098, 0.15294117647058825, 0.1568627450980392),# NK
    (1.0, 0.596078431372549, 0.5882352941176471),                 # Neuron
    (0.5803921568627451, 0.403921568627451, 0.7411764705882353),  # Plasma
    (0.7725490196078432, 0.6901960784313725, 0.8352941176470589),# T
    (0.5490196078431373, 0.33725490196078434, 0.29411764705882354)# cycling
]

In [ ]:
def label_fovs_by_module(df_fov, module_col, frac=0.1):
    """
    frac = fraction of FOVs to call 'high' and 'low' (e.g. 0.1 → top/bottom 10%)
    """
    df = df_fov.dropna(subset=[module_col]).copy()
    n = len(df)
    k = max(1, int(n * frac))
    
    df_sorted = df.sort_values(module_col, ascending=False)
    high_fovs = df_sorted.head(k)[["patient_sec", "fov"]].assign(group="high")
    low_fovs  = df_sorted.tail(k)[["patient_sec", "fov"]].assign(group="low")
    
    return pd.concat([high_fovs, low_fovs], ignore_index=True)


In [ ]:
df_z['fov']=[i.split('_')[1] for i in df_z['fov'].tolist()]

In [ ]:
def plot_module_composition_side_by_side(
    df_fov,
    combined_obs,
    module_col,
    xmax,
    desc,
    frac=0.1,
    celltype_general_col="cell_type_general",
    cellcategory_col="cell_category",
    figsize=(14,8),
    global_categories=None,
    global_palette=None
):
    # Optional color map
    color_map = None
    if global_categories is not None and global_palette is not None:
        color_map = dict(zip(global_categories, global_palette))

    # Prepare df
    df_mod = df_fov.dropna(subset=[module_col]).copy()
    df_mod = df_mod.sort_values(module_col, ascending=False)

    n = len(df_mod)
    k = max(1, int(n * frac))

    chosen_high = df_mod.head(k)[["patient_sec", "fov"]]
    chosen_low  = df_mod.tail(k)[["patient_sec", "fov"]]

    def compute_comp(chosen):
        merged = combined_obs.merge(chosen, on=["patient_sec", "fov"], how="inner")
        if merged.empty:
            return None
        
        per_fov_counts = (
            merged.groupby(["patient_sec", "fov", celltype_general_col, cellcategory_col])
            .size()
            .reset_index(name="count")
        )
        totals = (
            per_fov_counts.groupby(["patient_sec","fov"])["count"]
            .sum()
            .reset_index(name="total_count")
        )
        per_fov = per_fov_counts.merge(totals, on=["patient_sec","fov"], how="left")
        per_fov["fraction"] = per_fov["count"] / per_fov["total_count"]
        return per_fov

    per_fov_high = compute_comp(chosen_high)
    per_fov_low  = compute_comp(chosen_low)

    # order from overall mean
    all_per_fov = pd.concat([per_fov_high, per_fov_low], axis=0)
    mean_comp = (
        all_per_fov.groupby([celltype_general_col, cellcategory_col])["fraction"]
        .mean()
        .reset_index()
        .sort_values("fraction", ascending=False)
    )
    y_order = mean_comp[celltype_general_col].unique()

    # PLOT
    fig, (ax_left, ax_right) = plt.subplots(
        1, 2,
        figsize=figsize,
        sharey=True,
        gridspec_kw={'wspace': 0}   # REMOVE the middle space
    )

       # LEFT = LOW
    sns.barplot(
        data=per_fov_low,
        x="fraction",
        y=celltype_general_col,
        hue=cellcategory_col,
        palette=color_map,
        order=y_order,
        orient="h",
        dodge=False,
        estimator="mean",
        errorbar=("ci", 95),
        err_kws={"linewidth":1},
        ax=ax_left
    )
    ax_left.tick_params(axis='y', labelsize=8.5)
    ax_left.set_title("")
    ax_left.set_xlabel("Cell% in Low " + desc + "FOVs")   # ★ updated
    ax_left.set_ylabel("")

    # mirror the axis
    ax_left.set_xlim(xmax, 0)

    ax_left.tick_params(left=True)
    ax_left.legend_.remove()

    # RIGHT = HIGH
    sns.barplot(
        data=per_fov_high,
        x="fraction",
        y=celltype_general_col,
        hue=cellcategory_col,
        palette=color_map,
        order=y_order,
        orient="h",
        dodge=False,
        estimator="mean",
        errorbar=("ci", 95),
        err_kws={"linewidth":1},
        ax=ax_right
    )
    ax_right.set_title("")
    ax_right.set_xlabel("Cell% in High " + desc + "FOVs")   # ★ updated

    ax_right.set_xlim(0, xmax)

    # put y labels only on the right
    ax_right.yaxis.set_label_position("right")
    ax_right.yaxis.tick_right()

    # remove right-side ticks
    ax_right.tick_params(right=False)


    # -------------------------
    # Add proportion annotations outside bars
    # -------------------------
    
    def format_val(width):
        return f"{width:.2f}"
    left_offset  = xmax * 0.19      # move LEFT plot annotation further left
    right_offset = xmax * 0.11      # move RIGHT plot annotation closer to the bar


    # LEFT (LOW FOVs) — annotation goes to the LEFT of the bar
    for patch in ax_left.patches:
        width = patch.get_width()
        if np.isnan(width) or width <= 0:
            continue
        y = patch.get_y() + patch.get_height() / 2
    
        # because of reversed x-axis, larger x => further LEFT
        x_text = width + left_offset
    
        ax_left.text(
            x_text, y,
            format_val(width*100),
            va="center",
            ha="left",      # left-align so text extends further outward
            fontsize=7,
        )
    
    # RIGHT (HIGH FOVs) — annotation to the right of the bar (normal axis)
    for patch in ax_right.patches:
        width = patch.get_width()
        if np.isnan(width) or width <= 0:
            continue
        y = patch.get_y() + patch.get_height() / 2
    
        x_text = width + right_offset
    
        ax_right.text(
            x_text, y,
            format_val(width*100),
            va="center",
            ha="left",
            fontsize=7,
        )
    
            
    # shared legend
    handles, labels = ax_right.get_legend_handles_labels()
    ax_right.legend(
        handles, labels,
        title="Cell Category",
        bbox_to_anchor=(1.05, 1),
        loc="upper left"
    )

    plt.tight_layout()
    plt.show()


In [ ]:
plot_module_composition_side_by_side(
    df_fov=df_z,
    combined_obs=spot_anno,
    xmax=0.4,
    desc='Ag Pres/Treg ',
    celltype_general_col="network_anno",
    module_col="Ag Presentation/Treg",
    frac=0.05,
    figsize=(8.5,4.6),
    global_categories=global_categories,
    global_palette=global_palette
)


# fov vis

In [ ]:
df_z_sort = df_z.sort_values('Ag Presentation/Treg',ascending=False).reset_index()

In [ ]:
df_z_sort_sample = df_z_sort[df_z_sort['patient_sec']=='11808_A4_Cos3']
df_z_sort_sample

In [ ]:
top_fovs = df_z_sort_sample.nlargest(5, 'Ag Presentation/Treg').fov.tolist()
bottom_fovs =df_z_sort_sample.nsmallest(5, 'Ag Presentation/Treg').fov.tolist()

selected_fovs = set(top_fovs + bottom_fovs)
selected_fovs

In [ ]:
selected_fovs = {'106', '119', '12', '150', '151', '45', '69', '7', '82', '90'}

In [ ]:
prefix = '/path/to/cosmx_data/cd_resection/'

In [ ]:
adata = sc.read_h5ad(prefix+'Cos3_6KDC_strict11808_A4_0716_11_08_2025_14_19_48_615'+'/Processed/'+'norm_log1p.h5ad')

In [ ]:
idx = ['11808_A4_Cos3_'+i for i in adata.obs_names.tolist()]

In [ ]:
adata.obs['idx']=idx
adata.obs

In [ ]:
tmp = spot_anno[spot_anno['patient_sec']=='11808_A4_Cos3']
adata_obs_new =adata.obs.merge(tmp[['cell_category']], how = 'right',right_index=True, left_on = 'idx')

In [ ]:
adata_sub = adata[adata.obs.index.isin(adata_obs_new.index.tolist())]

In [ ]:
adata_sub.obs=adata_sub.obs.merge(adata_obs_new[['cell_category']], how = 'left', left_index=True, right_index=True)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np

coordinates = adata_sub.obsm['spatial_fov']          # shape (n_cells, 2)
category_column = 'cell_category'
categories = adata_sub.obs[category_column].astype('category')

palette = sns.color_palette('tab20', len(categories.cat.categories))
color_map = dict(enumerate(palette))
colors = categories.cat.codes.map(color_map)

# mask for selected vs non-selected FOVs
is_selected = adata_sub.obs['fov'].isin(selected_fovs).values

plt.figure(figsize=(9, 10))

# 1) plot all cells faintly as background
plt.scatter(
    coordinates[~is_selected, 0],
    coordinates[~is_selected, 1],
    c=colors[~is_selected],
    s=0.5,
    alpha=0.1
)

# 2) plot selected FOV cells more strongly
plt.scatter(
    coordinates[is_selected, 0],
    coordinates[is_selected, 1],
    c=colors[is_selected],
    s=1.5,
    alpha=0.9
)

ax = plt.gca()

plt.title("FOVs with Highest and Lowest Ag Pres/Treg Destab LRs \nCD_R1_B7 (Red = High, Blue = Low)")
plt.xlabel("X coordinate")
plt.ylabel("Y coordinate")

# legend for cell categories
handles = [
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=palette[i], markersize=8)
    for i in range(len(categories.cat.categories))
]
plt.legend(
    handles,
    categories.cat.categories,
    title=category_column,
    bbox_to_anchor=(1.05, 1),
    loc='upper left'
)

plt.tight_layout()
plt.show()


In [ ]:
adata_sub_fov = adata_sub[adata_sub.obs['fov'].isin(['42','43','44','45','46','47','48','49','50','51','52','53'])]
#adata_sub_fov=adata_sub_fov[~adata_sub_fov.obs['network_anno'].isin(['SELENOP Fib','Cycling'])]

In [ ]:
network_anno_simple = []
for i in adata_sub_fov.obs['network_anno'].tolist():
    if i in ['Mac S_M_S','Mac S_M_P']:
        network_anno_simple.append('Mac S')
    elif i == 'Neutrophil':
        network_anno_simple.append('Granulocyte')
    elif i == 'Inf Mo-Mac':
        network_anno_simple.append('Mo-Mac')
    elif 'Treg' in i:
        network_anno_simple.append('Treg')
    else:
        network_anno_simple.append(i)

In [ ]:
adata_sub_fov.obs['network_anno_simple']=network_anno_simple

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
adata_sub_fov=adata_sub_fov[~adata_sub_fov.obs['cell_category'].isin(['Cycling'])]
coordinates = adata_sub_fov.obsm['spatial_fov']
category_column = 'cell_category'

# Ensure category dtype
categories = adata_sub_fov.obs[category_column].astype('category')

# Build palette + mapping
palette = sns.color_palette('tab20', len(categories.cat.categories))
color_map = {code: palette[code] for code in range(len(categories.cat.categories))}
colors = categories.cat.codes.map(color_map)

fig, ax = plt.subplots(figsize=(3.5, 5))

# ---- KEY FIX: remove axes background completely ----
ax.set_facecolor("none")     # no axes background box
fig.patch.set_facecolor("none")  # no figure-level box

# ---- MAIN SCATTER ----
ax.scatter(
    coordinates[:, 0],
    coordinates[:, 1],
    c=colors,
    s=0.5,
    alpha=0.9
)

# ---- REMOVE ALL AXIS ELEMENTS ----
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title("")

# Remove spines (axis borders)
for spine in ax.spines.values():
    spine.set_visible(False)

# Remove axes entirely
ax.axis("off")

# ---- LEGEND ----
handles = [
    plt.Line2D([0], [0], marker='o', color='none',
               markerfacecolor=palette[i], markersize=8)
    for i in range(len(categories.cat.categories))
]

ax.legend(
    handles,
    categories.cat.categories,
    title="Cell Type",
    loc="upper center",
    bbox_to_anchor=(0.5, -0.02),  # center bottom, slightly below plot
    ncol=2,                       # increase if you want multiple columns
    frameon=False,
    handletextpad=0.4,
    columnspacing=0.8
)

plt.show()


In [ ]:
spot_lr_df['pat_sec'] = spot_lr_df.index.map(lambda s: '_'.join(s.split('_')[:-2]))
spot_lr_df_sub = spot_lr_df[spot_lr_df['pat_sec']=='11808_A4_Cos3']

In [ ]:
spot_lr_df_sub['idx'] = spot_lr_df_sub.index.map(
    lambda s: '_'.join(s.rsplit('_', 2)[1:])
)

In [ ]:
spot_lr_df_sub['fov']=[i.split('_')[1] for i in spot_lr_df_sub['idx'].tolist()]
spot_lr_df_sub

In [ ]:
spot_lr_df_sub2=spot_lr_df_sub[spot_lr_df_sub['fov'].isin(['42','43','44','45','46','47','48','49','50','51','52','53'])]

In [ ]:

spot_lr_sel_full = spot_lr_df_sub[[
'HLA-DRA^CD63',
 'HLA-DRA^CD81',
 'ICAM1^MSN',
 'ITGB2^CD226',
 'ICAM1^SPN',
 'PLA2G2A^ITGB1',
 'CD99^CD81',
 'CXCL14^CXCR4',
 'HLA-DRA^LAG3',
 'SIRPA^CD69',
 'CCL4^CCR5',
'idx']]
spot_lr_sel_full

In [ ]:
spot_lr_sel = spot_lr_df_sub2[[
'HLA-DRA^CD63',
 'HLA-DRA^CD81',
 'ICAM1^MSN',
 'ITGB2^CD226',
 'ICAM1^SPN',
 'PLA2G2A^ITGB1',
 'CD99^CD81',
 'CXCL14^CXCR4',
 'HLA-DRA^LAG3',
 'SIRPA^CD69',
 'CCL4^CCR5',
'idx']]
spot_lr_sel

In [ ]:
adata_sub_fov.obs=adata_sub_fov.obs.merge(spot_lr_sel, how = 'left', left_index=True, right_on = 'idx')


In [ ]:
adata_sub.obs=adata_sub.obs.merge(spot_lr_sel_full, how = 'left', left_index=True, right_on = 'idx')


In [ ]:
def add_module_scores_from_obs(
    adata,
    module_lrs,
    agg="mean",
    prefix="module_"
):
    """
    Compute per-spot (per-obs) module scores when LR pairs are stored in adata.obs.

    Parameters
    ----------
    adata : AnnData
        AnnData where LR scores are in .obs columns.
    module_lrs : dict
        {module_name: [lr_pair1, lr_pair2, ...]}
    agg : {'mean', 'sum'}
        How to aggregate LR scores within a module.
    prefix : str
        Prefix for the new obs column names.

    Returns
    -------
    adata : modified AnnData with new columns in .obs
    """
    for module_name, lr_list in module_lrs.items():

        # Intersect LR list with available columns
        lr_cols = [lr for lr in lr_list if lr in adata.obs.columns]
        if not lr_cols:
            print(f"[WARN] No LR pairs for module '{module_name}' found in obs.")
            continue

        X = adata.obs[lr_cols]

        if agg == "mean":
            adata.obs[f"{prefix}{module_name}"] = X.mean(axis=1)
        elif agg == "sum":
            adata.obs[f"{prefix}{module_name}"] = X.sum(axis=1)
        else:
            raise ValueError("agg must be 'mean' or 'sum'")

    return adata


In [ ]:
module_lrs = {
    "Ag Pres/Treg Destab": ['HLA-DRA^CD63',
 'HLA-DRA^CD81',
 'ICAM1^MSN',
 'ITGB2^CD226',
 'ICAM1^SPN',
 'PLA2G2A^ITGB1',
 'CD99^CD81',
 'CXCL14^CXCR4',
 'HLA-DRA^LAG3',
 'SIRPA^CD69',
 'CCL4^CCR5'],
}

add_module_scores_from_obs(adata_sub_fov, module_lrs)


In [ ]:
add_module_scores_from_obs(adata_sub, module_lrs)

In [ ]:
import matplotlib.patches as patches
import numpy as np

coordinates = adata_sub_fov.obsm['spatial_fov']
value_col = 'module_Ag Pres/Treg Destab'  # continuous column in .obs

# If it's not yet in .obs, see note below

vals = adata_sub_fov.obs[value_col].astype(float).values

plt.figure(figsize=(3.5, 5.2))
ax = plt.gca()

# ---------- MAIN SCATTER: continuous Reds ----------
sc = ax.scatter(
    coordinates[:, 0],
    coordinates[:, 1],
    c=vals,
    cmap='Reds',
    s=0.5
)

# ---------- REMOVE ALL AXIS ELEMENTS ----------
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title("")
for spine in ax.spines.values():
    spine.set_visible(False)

# Lock limits before drawing rectangles
xlim = ax.get_xlim()
ylim = ax.get_ylim()

# Restore limits so rectangle is preserved
ax.set_xlim(xlim)
ax.set_ylim(ylim)

# ---------- COLORBAR FOR CONTINUOUS VALUES ----------
cbar = plt.colorbar(
    sc,
    ax=ax,
    orientation="horizontal",
    fraction=0.06,   # thickness
    pad=0.08         # space from plot
)

cbar.set_label("Ag Pres/Treg Destab LR Score", labelpad=6)

# ---------- SAVE ----------
plt.savefig(
    prefix + 'Cos3_6KDC_strict11808_A4_0716_11_08_2025_14_19_48_615'
    + '/Processed/anno_42-53_Ag Pres_Treg Destab_Reds_box.pdf',
    dpi=1000,
    bbox_inches="tight"
)
plt.show()


In [ ]:
import matplotlib.patches as patches
import numpy as np

coordinates = adata_sub.obsm['spatial_fov']
value_col = 'module_Ag Pres/Treg Destab'  # continuous column in .obs

# If it's not yet in .obs, see note below

vals = adata_sub.obs[value_col].astype(float).values

plt.figure(figsize=(7,10))
ax = plt.gca()

# ---------- MAIN SCATTER: continuous Reds ----------
sc = ax.scatter(
    coordinates[:, 0],
    coordinates[:, 1],
    c=vals,
    cmap='Reds',
    s=0.5
)

# ---------- REMOVE ALL AXIS ELEMENTS ----------
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title("")
for spine in ax.spines.values():
    spine.set_visible(False)

# Lock limits before drawing rectangles
xlim = ax.get_xlim()
ylim = ax.get_ylim()

# Restore limits so rectangle is preserved
ax.set_xlim(xlim)
ax.set_ylim(ylim)

# ---------- COLORBAR FOR CONTINUOUS VALUES ----------
cbar = plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label(value_col, rotation=90) 

# ---------- SAVE ----------
plt.savefig(
    prefix + 'Cos3_6KDC_strict11808_A4_0716_11_08_2025_14_19_48_615'
    + '/Processed/anno_Ag Pres_Treg Destab_Reds_box.pdf',
    dpi=1000,
    bbox_inches="tight"
)
plt.show()


# concatenate for graph

In [ ]:

def concatenate_fov_X_per_sample(
    base_dir,
    sample_folders,
    output_suffix='_liana_lrspot_X_combined.parquet',
):
    """
    For each sample, concatenate X matrix (spot x LR) from all FOV h5ad files
    in the liana folder and save as a single parquet file.

    Each row = one obs (spot/cell) from a specific FOV and sample.
    Columns = LR features (adata.var_names) + metadata columns: ['fov', 'sample'].
    Index = original obs_names.
    """
    
    for sample in sample_folders:
        print(f"\nProcessing sample: {sample}")
        
        liana_path = os.path.join(base_dir, sample, "Processed", "liana")
        output_path = os.path.join(base_dir, sample, "Processed", f"{sample}{output_suffix}")
        
        # Get all h5ad files in liana directory
        files = [f for f in os.listdir(liana_path) if f.endswith('.h5ad')]
        
        print(f"  Found {len(files)} FOV h5ad files")
        
        try:
            df_list = []
            for i, file in enumerate(files, 1):
                file_path = os.path.join(liana_path, file)
                
                # Read full AnnData (we need X in memory to call .toarray())
                adata = ad.read_h5ad(file_path)
                
                # Convert X to DataFrame: rows = obs, cols = var
                # (if X is already dense, .toarray() will still work)
                X_df = pd.DataFrame(
                    adata.X.toarray(),
                    index=adata.obs_names,
                    columns=adata.var_names,
                )
                
                # Add FOV + sample identifiers
                fov_id = os.path.splitext(file)[0]
                X_df['fov'] = fov_id
                X_df['sample'] = sample

                df_list.append(X_df)
                
                # Clean up
                del adata, X_df
                gc.collect()
                
                if i % 10 == 0:
                    print(f"    Loaded {i}/{len(files)} files...")
            
            # Concatenate all FOV DataFrames for this sample
            print(f"  Concatenating {len(df_list)} dataframes...")
            sample_combined = pd.concat(df_list, axis=0, ignore_index=False)
            
            # Optional: ensure a consistent index name
            sample_combined.index.name = "obs_id"
            
            # Save the combined sample data
            sample_combined.to_parquet(output_path)
            print(f"  ✓ Saved to {output_path}")
            print(f"    Shape: {sample_combined.shape}")
            
            # Clean up
            del df_list, sample_combined
            gc.collect()
            
        except Exception as e:
            print(f"  ✗ Error processing {sample}: {e}")
            gc.collect()


def concatenate_all_samples_X(
    base_dir,
    sample_folders,
    output_suffix='_liana_lrspot_X_combined.parquet',
    final_output='all_samples_liana_lrspot_X_combined.parquet',
):
    """
    Concatenate all sample-level combined X (spot × LR) files into one final file.

    - Expects each sample file to already contain 'sample' and 'fov' columns.
    - After concatenation, sets a MultiIndex that includes the sample name so
      the index encodes which sample each row came from.
    """
    print(f"\nConcatenating all {len(sample_folders)} samples...")
    
    all_samples_list = []
    
    for i, sample in enumerate(sample_folders, 1):
        sample_file = os.path.join(base_dir, sample, "Processed", f"{sample}{output_suffix}")
        
        df = pd.read_parquet(sample_file)
        all_samples_list.append(df)
        
        print(f"  Loaded {i}/{len(sample_folders)}: {sample} (shape: {df.shape})")
        
        if i % 10 == 0:
            gc.collect()
    
    print(f"\nConcatenating {len(all_samples_list)} samples...")
    final_combined = pd.concat(all_samples_list, axis=0, ignore_index=False)
    
    # Make sure we have 'sample' in the index as requested
    # Keep existing index (obs_id) but prepend sample to make a MultiIndex
    if final_combined.index.name is None:
        final_combined.index.name = "obs_id"
    original_index_name = final_combined.index.name
    
    final_combined = final_combined.reset_index()  # bring obs_id out as a column
    # Now set a MultiIndex with sample + obs_id (and keep fov as a column)
    final_combined.set_index(['sample', original_index_name], inplace=True)
    
    # Save final combined file
    final_output_path = os.path.join(base_dir, final_output)
    final_combined.to_parquet(final_output_path)
    
    print(f"✓ Final combined file saved to {final_output_path}")
    print(f"  Final shape: {final_combined.shape}")
    print(f"  Final index names: {final_combined.index.names}")
    
    del all_samples_list
    gc.collect()
    
    return final_combined


In [ ]:

# Usage
base_dir = '/path/to/cosmx_data/cd_resection'
sample_folders = sample_list

# Step 1: Concatenate FOV .var from h5ad files within each sample
concatenate_fov_X_per_sample(base_dir, sample_folders)


In [ ]:
lr_x_df = concatenate_all_samples_X(base_dir,sample_folders)

In [ ]:
spot_lr_df_filled = lr_x_df.fillna(0.0)

In [ ]:
spot_lr_df_filled['patient_sec'] = spot_lr_df_filled.index.get_level_values('sample').map(comb_lst)

In [ ]:
spot_lr_df_filled['cell'] = spot_lr_df_filled.index.get_level_values('obs_id')

In [ ]:
spot_lr_df_filled['new_idx']=[i+'_'+j for i,j in zip(spot_lr_df_filled['patient_sec'].tolist(), spot_lr_df_filled['cell'].tolist())]


In [ ]:
spot_lr_df_filled.index = spot_lr_df_filled['new_idx'].tolist()

In [ ]:
spot_lr_df_filled=spot_lr_df_filled.drop(['patient_sec','cell','new_idx','fov'],axis = 1)

In [ ]:
spot_lr_df=spot_lr_df_filled
spot_lr_df

In [ ]:
spot_anno =pd.read_parquet('/path/to/cosmx_data/cd_resection/combined/all_samples.parquet')
spot_anno

In [ ]:
spot_anno=spot_anno[['fov','source_file', 'cell_type', 'cell_type_short', 'cell_category',
       'cell_type_general', 'patient_sec', 'patient']]

In [ ]:
spot_anno['new_idx']=[i+'_'+j for i,j in zip(spot_anno['patient_sec'].tolist(), spot_anno.index.tolist())]

spot_anno.index =spot_anno['new_idx'].tolist()
spot_anno=spot_anno.drop('new_idx',axis = 1)

In [ ]:
spot_anno=spot_anno[spot_anno.index.isin(spot_lr_df.index.tolist())]

In [ ]:
final_df['function'] = final_df.index.map(lr_function)

In [ ]:
final_df['function'].value_counts()

In [ ]:
lr_meta_df=final_df[['ligand', 'receptor','function']]

In [ ]:
lr_meta_df=lr_meta_df.drop_duplicates()

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx

def build_celltype_lr_graphs_shared_lr_efficient(
    spot_lr_df: pd.DataFrame,
    spot_annot: pd.DataFrame,
    lr_meta_df: pd.DataFrame,
    cell_type_col: str = "cell_type_general",
    lr_pair_col: str = "lr_pair",
    module_col: str = "module",
):
    """
    Build an undirected cell–cell graph per LR module based on shared LR activity,
    in a memory-efficient way (no melt / huge long table).

    For each module:
      1) For each cell type × LR pair, compute the mean LR score across spots.
      2) Create a matrix M (cell_types × LR_pairs_in_module).
      3) Compute W = M @ M.T:
            W[A,B] = sum_LR mean_score[A,LR] * mean_score[B,LR]
      4) Build an undirected graph with:
            nodes = cell types
            edges = W[A,B] > 0
            edge weight = W[A,B]
    """

    # 0) sanity check
    if not spot_lr_df.index.equals(spot_annot.index):
        raise ValueError("spot_lr_df and spot_annot must share the same index (spots).")

    # 1) restrict lr_meta_df to LR pairs present in spot_lr_df
    lr_meta_sub = lr_meta_df[lr_meta_df[lr_pair_col].isin(spot_lr_df.columns)].copy()
    if lr_meta_sub.empty:
        raise ValueError("No overlap between lr_meta_df[lr_pair_col] and spot_lr_df columns.")

    if module_col not in lr_meta_sub.columns:
        raise ValueError(f"Column '{module_col}' not found in lr_meta_df.")

    # 2) cell type labels as categorical → fast integer codes
    ct_series = spot_annot[cell_type_col].astype("category")
    cell_types = list(ct_series.cat.categories)   # e.g. ['Treg_Early','Treg_Late','CD9+ Mac',...]
    ct_codes = ct_series.cat.codes.to_numpy()     # shape: (n_spots,)

    graphs_by_module = {}

    # 3) loop over modules
    for module_name, lr_rows in lr_meta_sub.groupby(module_col):
        if module_name == "other":
            # skip the huge 'other' bucket if you want
            print(f"Skipping module '{module_name}'")
            continue

        lr_list = [lr for lr in lr_rows[lr_pair_col] if lr in spot_lr_df.columns]
        print(lr_list)
        if len(lr_list) == 0:
            continue

        print(f"Processing module '{module_name}' with {len(lr_list)} LR pairs...")

        # 3a) subset LR matrix for this module and fill NaN → 0 (= no interaction)
        # NOTE: this is still n_spots × (#LR in module). Cast to float32 to save RAM.
        module_mat = spot_lr_df[lr_list].fillna(0.0).to_numpy(dtype=np.float32)

        n_ct = len(cell_types)
        n_lr = len(lr_list)

        # 3b) build M: cell_types × LR_pairs_in_module
        M = np.zeros((n_ct, n_lr), dtype=np.float32)

        for i, ct in enumerate(cell_types):
            mask = (ct_codes == i)
            if not np.any(mask):
                continue
            # mean across spots belonging to this cell type
            M[i, :] = module_mat[mask, :].mean(axis=0)

        # if everything is zero, skip
        if not np.any(M):
            print(f"  Module '{module_name}': all zeros after aggregation, skipping.")
            continue

        # 3c) compute W = M @ M.T (n_ct × n_ct)
        W = M @ M.T

        # 4) build graph
        G = nx.Graph()
        for ct in cell_types:
            G.add_node(ct, type="cell_type", module=module_name)

        # add edges for W[i,j] > 0
        for i in range(n_ct):
            for j in range(i + 1, n_ct):
                w = float(W[i, j])
                if w <= 0:
                    continue
                G.add_edge(
                    cell_types[i],
                    cell_types[j],
                    weight=w,
                    shared_lr_strength=w,
                    module=module_name
                )

        print(f"  Finished module '{module_name}': {G.number_of_nodes()} nodes, {G.number_of_edges()} edges.")
        graphs_by_module[module_name] = G

    return graphs_by_module


In [ ]:
spot_anno = spot_anno.reindex(spot_lr_df.index)

In [ ]:
tregs = pd.read_csv("/path/to/cosmx_data/cd_resection/combined/strict_treg_pt_transfer_labels.csv")
tregs

In [ ]:
tregs['early_late']=['Late Treg' if i>=3 else 'Early Treg' for i in tregs['predicted.id'].tolist()]
tregs.index = tregs['Unnamed: 0'].tolist()
tregs

In [ ]:
spot_anno = spot_anno.merge(tregs[['early_late']],how='left',left_index=True, right_index=True)
spot_anno['early_late'].value_counts()

In [ ]:
network_anno =[]
for i,j,k,m in zip (spot_anno['cell_category'].tolist(),spot_anno['cell_type_general'].tolist(),spot_anno['cell_type_short'].tolist(),spot_anno['early_late'].tolist()):
    if i == 'Myeloid':
        if 'DC' in j:
            network_anno.append(j)
        elif k in ['Mac S+XS-','Mac S+SG+']:
            network_anno.append('Mac S')
        elif k == 'Mac S+M+P+':
            network_anno.append('Mac S_M_P')
        elif k == 'Mac S+M+S+':
            network_anno.append('Mac S_M_S')
        elif 'CD9' in j:
            network_anno.append('CD9 Mac')
        else:
            network_anno.append(k)
    elif i == 'Connective':
        if 'OGN' in j:
            network_anno.append('OGN_RSPO3 Fib')
        elif 'ADAMDEC1' in j:
            network_anno.append('ADAMDEC1 Fib')
        elif 'PLA2G2A' in j:
            network_anno.append('PLA2G2A Fib')
        elif 'SELENOP' in j:
            network_anno.append('SELENOP Fib')
        else: 
            network_anno.append(j)
    elif i == 'T':
        if j == 'Treg':
            if pd.notna(m):
                network_anno.append(m)
            else:
                network_anno.append('drop')
        elif 'Tfh' in j:
            network_anno.append('Tfh')
        else:
            network_anno.append(j)
    else:
        network_anno.append(i)

In [ ]:
spot_anno['network_anno']=network_anno
spot_anno['network_anno'].value_counts()

In [ ]:
spot_anno = spot_anno[spot_anno['network_anno']!='drop']

In [ ]:
spot_lr_df=spot_lr_df[spot_lr_df.index.isin(spot_anno.index.tolist())]

In [ ]:
lr_meta_df['lr_pair']=lr_meta_df.index.tolist()
lr_meta_df['module']=lr_meta_df['function'].tolist()

In [ ]:
lr_meta_df.to_parquet('/path/to/cosmx_data/cd_resection/combined/lr_meta_df_new.parquet')

In [ ]:
spot_lr_df.to_parquet('/path/to/cosmx_data/cd_resection/combined/spot_lr_df.parquet')

In [ ]:
spot_anno.to_parquet('/path/to/cosmx_data/cd_resection/combined/spot_anno.parquet')

In [ ]:
spot_anno.to_parquet('/path/to/cosmx_data/cd_resection/combined/spot_anno_early_late.parquet')

In [ ]:
lr_meta_df['function'].value_counts()

In [ ]:
graphs_by_module = build_celltype_lr_graphs_shared_lr_efficient(
    spot_lr_df=spot_lr_df,
    spot_annot=spot_anno,
    lr_meta_df=lr_meta_df,
    cell_type_col="network_anno",
    lr_pair_col="lr_pair",
    module_col="module",
)

In [ ]:
ct_mapper = spot_anno[['cell_category','network_anno']].drop_duplicates()
ct_mapper

In [ ]:
import pandas as pd
import networkx as nx
import os
import re

def clean_name(name):
    """
    Make any node or module name Cytoscape-safe:
    - Replace spaces -> _
    - Remove any weird characters
    """
    name = str(name)
    name = name.replace(" ", "_")
    # Remove anything not alphanumeric or underscore
    name = re.sub(r"[^A-Za-z0-9_]", "", name)
    return name

def export_graph_to_csv(G, module_name, ct_mapper, outdir="./cytoscape_exports"):
    """
    G : networkx graph
    module_name : name of module (used for file naming)
    ct_mapper : DataFrame containing:
                 - 'network_anno' (matching G node names)
                 - 'cell_category' (category to export)
    """

    os.makedirs(outdir, exist_ok=True)

    # Clean filename
    module_name_clean = clean_name(module_name)

    # Ensure ct_mapper is DataFrame with required columns
    if not isinstance(ct_mapper, pd.DataFrame):
        raise ValueError("ct_mapper must be a DataFrame with 'network_anno' and 'cell_category' columns.")

    if "network_anno" not in ct_mapper.columns or "cell_category" not in ct_mapper.columns:
        raise ValueError("ct_mapper must contain 'network_anno' and 'cell_category' columns.")

    # Make a lookup dictionary for fast matching
    category_lookup = dict(zip(ct_mapper["network_anno"], ct_mapper["cell_category"]))

    # ---------------------
    # 1. NODES
    # ---------------------
    node_rows = []
    for n, data in G.nodes(data=True):
        clean_id = clean_name(n)

        # Lookup cell_category using 'network_anno'
        cell_cat = category_lookup.get(n, None)

        row = {
            "id": clean_id,     # safe Cytoscape ID
            "label": n,         # pretty label
            "cell_category": cell_cat
        }
        row.update(data)
        node_rows.append(row)

    nodes_df = pd.DataFrame(node_rows)

    # ---------------------
    # 2. EDGES
    # ---------------------
    edge_rows = []
    for u, v, data in G.edges(data=True):
        row = {
            "source": clean_name(u),
            "target": clean_name(v),
        }
        row.update(data)
        edge_rows.append(row)

    edges_df = pd.DataFrame(edge_rows)

    # ---------------------
    # 3. Save files
    # ---------------------
    nodes_path = os.path.join(outdir, f"{module_name_clean}_nodes_new.csv")
    edges_path = os.path.join(outdir, f"{module_name_clean}_edges_new.csv")

    nodes_df.to_csv(nodes_path, index=False)
    edges_df.to_csv(edges_path, index=False)

    print(f"Saved module '{module_name}' → {module_name_clean}")
    print("  Nodes:", nodes_path)
    print("  Edges:", edges_path)


In [ ]:
for module_name, G in graphs_by_module.items():
    # Optional: drop super-weak edges before export
    # e.g., keep only top edges
    # threshold = np.percentile([d["weight"] for _,_,d in G.edges(data=True)], 75)
    # edges_to_remove = [(u,v) for u,v,d in G.edges(data=True) if d["weight"] < threshold]
    # G_filtered = G.copy()
    # G_filtered.remove_edges_from(edges_to_remove)

    export_graph_to_csv(G, module_name, ct_mapper,outdir="/path/to/cosmx_data/cd_resection/combined/cytoscape_exports")


In [ ]:
edges_ref=pd.read_csv('/path/to/cosmx_data/cd_resection/combined/cytoscape_exports/Ag_PresentationTreg_edges.csv')

In [ ]:
edges_ref[edges_ref['weight']>=0.14].sort_values('weight')

In [ ]:
nodes=pd.read_csv('/path/to/cosmx_data/cd_resection/combined/cytoscape_exports/Ag_PresentationTreg_nodes_new.csv')
edges=pd.read_csv('/path/to/cosmx_data/cd_resection/combined/cytoscape_exports/Ag_PresentationTreg_edges_new.csv')

In [ ]:
edges[edges['weight']>=0.18].sort_values('weight')

In [ ]:
import pandas as pd
import networkx as nx

def compute_degrees_from_edges(
    edge_df,
    source_col="source",
    target_col="target",
    weight_col=None,
    min_weight=None,
    directed=False,
    remove_nodes=None,   # ← NEW ARGUMENT
):
    """
    Compute node degrees after filtering edges and optionally removing nodes.

    Parameters
    ----------
    edge_df : pd.DataFrame
        Edge table with at least source and target columns.
    source_col, target_col : str
        Column names for source and target nodes.
    weight_col : str or None
        Column with edge weight (for weighted degree). If None, only unweighted degree is computed.
    min_weight : float or None
        If given and weight_col is not None, keeps only edges with weight >= min_weight.
    directed : bool
        If True, build a directed graph and also return in/out degrees.
    remove_nodes : list or None
        Optional: list of node names to remove before computing degrees.

    Returns
    -------
    degree_df : pd.DataFrame
        Index = node IDs, columns = degree (and optionally weighted_degree, in_degree, out_degree, etc.).
    """

    df = edge_df.copy()

    # Weight filtering
    if weight_col is not None and min_weight is not None:
        df = df[df[weight_col] >= min_weight]

    # Build graph
    G = nx.DiGraph() if directed else nx.Graph()

    # Add edges
    if weight_col is not None:
        edges = df[[source_col, target_col, weight_col]].to_records(index=False)
        G.add_weighted_edges_from(edges)
    else:
        edges = df[[source_col, target_col]].to_records(index=False)
        G.add_edges_from(edges)

    # Remove user-specified nodes
    if remove_nodes is not None:
        # Only remove nodes that actually exist
        nodes_to_remove = [n for n in remove_nodes if n in G.nodes]
        G.remove_nodes_from(nodes_to_remove)

    # --- Degrees ---
    deg = pd.Series(dict(G.degree()), name="degree")

    if weight_col is not None:
        wdeg = pd.Series(dict(G.degree(weight="weight")), name="weighted_degree")
        degree_df = pd.concat([deg, wdeg], axis=1)
    else:
        degree_df = deg.to_frame()

    if directed:
        indeg = pd.Series(dict(G.in_degree()), name="in_degree")
        outdeg = pd.Series(dict(G.out_degree()), name="out_degree")
        degree_df = pd.concat([degree_df, indeg, outdeg], axis=1)

    degree_df.index.name = "node"
    return degree_df


In [ ]:

# Example: only edges with weight >= 0.2 in the "fibrotic" or "inflammatory" modules
degree_df = compute_degrees_from_edges(
    edges,
    source_col="source",
    target_col="target",
    weight_col="weight",
    min_weight=0.18,
    remove_nodes=["Cycling","SELENOP_Fib"]
)

degree_df


In [ ]:
nodes = nodes.drop('type',axis = 1)
nodes=nodes.merge(degree_df, how = 'right', left_on = 'id', right_index=True)

In [ ]:
nodes.to_csv('/path/to/cosmx_data/cd_resection/combined/cytoscape_exports/Ag_PresentationTreg_nodes_new.csv', index=False)

In [ ]:
edges_imm = pd.read_csv('/path/to/cosmx_data/cd_resection/combined/cytoscape_exports/ProInflammatory_edges_new.csv')
edges_inf = pd.read_csv('/path/to/cosmx_data/cd_resection/combined/cytoscape_exports/Immunomodulatory_edges_new.csv')
edges_fib = pd.read_csv('/path/to/cosmx_data/cd_resection/combined/cytoscape_exports/ProFibroticECM_edges_new.csv')
edges_ag = pd.read_csv('/path/to/cosmx_data/cd_resection/combined/cytoscape_exports/Ag_PresentationTreg_edges_new.csv')

In [ ]:
edges_all = pd.concat([edges_imm,edges_inf,edges_fib,edges_ag])

In [ ]:
# edges_all has: source, target, weight, module

top_edges = (
    edges_all
    .sort_values("weight", ascending=False)
    .groupby("module")
    .first()        # top row per module
    .reset_index()
)

top_edges


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Define custom palette mapping
module_palette = {
    "Pro-Fibrotic/ECM": "#ffe059",
    "Immunomodulatory": "#91a3ce",
    "Pro-Inflammatory": "#83cfb7",
    "Ag Presentation/Treg": "#c2c2c2"
}

plt.figure(figsize=(4.5,4))
sns.violinplot(
    data=edges_all, 
    x="module", 
    y="weight",
    palette=module_palette,    # <-- custom colors here
    inner="box", 
    cut=0
)
plt.xticks(rotation=30)
plt.title("")
plt.ylabel("Edge weight (LR strength btwn cells)")
plt.xlabel("LR Module")
plt.tight_layout()
plt.show()
